# VisionIST — SIFT examples

Two ways to get SIFT out of the VisionIST fleet, against boxes that are already running
somewhere else on the network. Nothing is installed here beyond `visionist-client`.

| Example | Box | Host | What it answers |
|---|---|---|---|
| 1 | `features` | `192.168.14.24` | *give me this image's SIFT keypoints*, in the SIFT-Extractor `(2+128) x N` layout |
| 2 | `opencv` | `10.16.144.115` | *match these two images* — SuperPoint + LightGlue, with a RANSAC fundamental matrix |

Both boxes speak the same envelope, so the only thing that differs between them is the
address and the name of the config section.


## Setup

```bash
pip install visionist-client opencv-python matplotlib scipy
```
### In this server the client is already installed

**We encourage students to deploy the features box in their own computer. It is very light. The opencv box is also very light provided you don't call the superglue (a deep learning model) - See example 2 below**

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from visionist_client import Visionist

# --- where the boxes live -------------------------------------------------
# Each box owns a permanent host port, issued once in the registry's ports.json.
FEATURES_HOST = "192.168.14.24:9071"     # features box
OPENCV_HOST   = "10.16.144.115:9065"     # opencv box (LightGlue build)

# If a box was started on its own with `-p 8061:8061`, use ":8061" instead.

features = Visionist(FEATURES_HOST)
opencv   = Visionist(OPENCV_HOST)

for name, box in [("features", features), ("opencv", opencv)]:
    print(f"{name:9s} ->", box.info())

### The two images

Point these at your own pair. The fallback is the fixture pair that ships with the
registry (`VisionIST_Library/boxes/opencv/test/`) — two views of the same scene.


In [ ]:
IMG_A = Path("/DATASETS/other/odivelas/L1152433.JPG")
IMG_B = Path("/DATASETS/other/odivelas/L1152432.JPG")

for p in (IMG_A, IMG_B):
    assert p.exists(), f"missing {p} — point IMG_A / IMG_B at your own pair"

print(IMG_A.resolve())
print(IMG_B.resolve())

---

## Example 1 — SIFT keypoints from the `features` box

The `features` box is a port of the old SIFT-Extractor command-line tool: same detector,
same output layout. It is pure CPU and holds no state.

- `keypoints` comes back as one `(num_images, 2 + 128, N)` array — **row 0 = x, row 1 = y,
  rows 2..129 = the 128-dimensional descriptor**.
- Images rarely yield the same number of keypoints, so the array is zero-padded on the last
  axis. **Always slice with `counts` first.**


In [ ]:
res = features.run(
    data   = {"images": [IMG_A, IMG_B],
              "names":  [IMG_A.name, IMG_B.name]},
    config = {"features": {"command": "extract",
                           "parameters": {"nfeatures": 1000,   # 0 keeps every keypoint
                                          "draw": True,        # red-dot annotated JPEGs
                                          "mat": True}}},      # one .mat per image
)

info = res.config["features"]
print(info["status"], "|", info["feature"],
      "| keypoints per image:", info["num_keypoints"],
      f"| {info['runtime']:.2f}s")

In [ ]:
kp     = res.keypoints          # (2, 130, N) — already decoded to np.ndarray
counts = res.counts             # (2,) int32 — the true N per image

xy_a   = kp[0, :2, :counts[0]].T     # (N_a, 2)   (x, y)
desc_a = kp[0, 2:, :counts[0]].T     # (N_a, 128) SIFT descriptors
xy_b   = kp[1, :2, :counts[1]].T
desc_b = kp[1, 2:, :counts[1]].T

print("image A:", xy_a.shape, desc_a.shape)
print("image B:", xy_b.shape, desc_b.shape)
print("first keypoint of A:", xy_a[0], "| descriptor norm:", np.linalg.norm(desc_a[0]).round(1))

The box also drew the keypoints for you — one JPEG per image, a filled red dot at every
keypoint, byte-for-byte what the original command-line tool produced.

In [ ]:
import io
from PIL import Image

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, jpg, name, n in zip(axes, res.annotated, [IMG_A.name, IMG_B.name], counts):
    ax.imshow(Image.open(io.BytesIO(jpg)))
    ax.set_title(f"{name} — {n} SIFT keypoints")
    ax.axis("off")
plt.tight_layout()
plt.show()

And one MATLAB `.mat` per image, key `kp`, a `130 x N` matrix — the same thing the
command-line tool wrote, so old MATLAB scripts keep working unchanged:

```matlab
load('00.mat')        % kp: 130 x N
xy   = kp(1:2, :);    % coordinates
desc = kp(3:end, :);  % descriptors
```


In [ ]:
for name, blob in zip(res.config["features"]["files"], res.mat):
    Path(name).write_bytes(blob)
    print("wrote", name, f"({len(blob)/1024:.0f} KB)")

# read one back here, just to show it round-trips
from scipy.io import loadmat
kp_mat = loadmat(res.config["features"]["files"][0])["kp"]
print("kp from .mat:", kp_mat.shape)

---

## Example 2 — matching with the `opencv` box, LightGlue path

The `opencv` box answers a different question: *which point in A is which point in B?*
Send it **exactly two** images and it does three things in one call — detect, match,
and throw out the pairs that are geometrically impossible.

`feature_extractor` picks the route:

| value | detector | matcher | needs |
|---|---|---|---|
| `SIFT` (default) | OpenCV SIFT | FLANN + Lowe ratio | nothing, pure CPU |
| `ORB` | OpenCV ORB | FLANN | nothing, pure CPU |
| `SUPERPOINT` | SuperPoint | **LightGlue** | a LightGlue build; GPU if available |
| `DISK` | DISK | **LightGlue** | same |

`ratio_thresh` is the FLANN path only. `device` is the LightGlue path only — leave it out
and the box picks CUDA if it can see a GPU, else CPU, and parks the models back on CPU
after about a minute idle.


In [ ]:
res = opencv.run(
    data   = {"images": [IMG_A, IMG_B]},            # exactly two for LightGlue
    config = {"opencv": {"command": "match",
                         "parameters": {"feature_extractor": "SUPERPOINT",
                                        "max_keypoints": 2048,
                                        # "device": "cuda:0",   # optional override
                                        }}},
)

info = res.config["opencv"]
print(info)        # status / matcher / feature_extractor / device / num_inliers / runtime

If this answers `status: "error"` with *LightGlue is not available in this image build*,
the box is running an image built without LightGlue. The SIFT path below still works on
the same box — it needs nothing extra.

In [ ]:
pts_a = res.matches_inliers_a     # (K, 2) points in image A
pts_b = res.matches_inliers_b     # (K, 2) the same points in image B
F     = res.fundamental_matrix    # (3, 3) — empty (0, 0) if RANSAC had too few matches

print("inlier correspondences:", pts_a.shape[0])
print("fundamental matrix:\n", F)

# descriptors on the LightGlue path are a zeros placeholder, not real descriptors
print("keypoints:", res.keypoints.shape, "| descriptors:", res.descriptors.shape, "(placeholder)")

### Look at the matches

`pts_a[i]` and `pts_b[i]` are the same physical point seen twice. Lines that run roughly
parallel mean the match is sound; a spray of crossing lines means it is not.

In [ ]:
import cv2

def draw_matches(path_a, path_b, pts_a, pts_b, every=1, title=""):
    a = cv2.cvtColor(cv2.imread(str(path_a)), cv2.COLOR_BGR2RGB)
    b = cv2.cvtColor(cv2.imread(str(path_b)), cv2.COLOR_BGR2RGB)
    h = max(a.shape[0], b.shape[0])
    canvas = np.zeros((h, a.shape[1] + b.shape[1], 3), dtype=np.uint8)
    canvas[:a.shape[0], :a.shape[1]] = a
    canvas[:b.shape[0], a.shape[1]:] = b
    off = a.shape[1]

    plt.figure(figsize=(16, 8))
    plt.imshow(canvas)
    for (xa, ya), (xb, yb) in zip(pts_a[::every], pts_b[::every]):
        plt.plot([xa, xb + off], [ya, yb], linewidth=0.6, alpha=0.7)
    plt.scatter(pts_a[::every, 0], pts_a[::every, 1], s=4, c="yellow")
    plt.scatter(pts_b[::every, 0] + off, pts_b[::every, 1], s=4, c="yellow")
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

draw_matches(IMG_A, IMG_B, pts_a, pts_b, every=4,
             title=f"{info['matcher']} — {info['num_inliers']} inliers (every 4th shown)")

### Sanity-check the geometry

The fundamental matrix encodes the relationship between the two viewpoints: a point in A
must lie on a line in B. For a correct `F`, the algebraic residual `x_b' F x_a` is near
zero for every inlier.

In [ ]:
if F.size:
    ones = np.ones((len(pts_a), 1))
    ha, hb = np.hstack([pts_a, ones]), np.hstack([pts_b, ones])
    residual = np.abs(np.einsum("ij,jk,ik->i", hb, F, ha))
    print(f"epipolar residual   mean {residual.mean():.4f}   median {np.median(residual):.4f}   max {residual.max():.4f}")
else:
    print("no fundamental matrix — too few matches")

---

## Same box, classical route — SIFT + FLANN

One parameter apart. Useful as a baseline, and it needs no GPU and no LightGlue build.

In [ ]:
res_sift = opencv.run(
    data   = {"images": [IMG_A, IMG_B]},
    config = {"opencv": {"command": "match",
                         "parameters": {"feature_extractor": "SIFT",
                                        "max_keypoints": 2048,
                                        "ratio_thresh": 0.85}}},
)
sift_info = res_sift.config["opencv"]
print(sift_info["matcher"], "|", sift_info["num_inliers"], "inliers",
      f"| {sift_info['runtime']:.2f}s")

draw_matches(IMG_A, IMG_B, res_sift.matches_inliers_a, res_sift.matches_inliers_b, every=4,
             title=f"{sift_info['matcher']} — {sift_info['num_inliers']} inliers (every 4th shown)")

In [ ]:
print(f"{'route':28s} {'inliers':>8s} {'runtime':>9s}")
for r in (res, res_sift):
    i = r.config["opencv"]
    print(f"{i['matcher']:28s} {i['num_inliers']:8d} {i['runtime']:8.2f}s")

---

## Notes

- **Ports are permanent.** `features` answers on 9071 and `opencv` on 9065 in every fleet;
  a partial fleet has gaps, never renumbered boxes. The ledger is
  [`registry/ports.json`](https://github.com/sipg-isr/VisionIST_Library/blob/main/registry/ports.json).
- **The config key is the box's name.** Get it wrong and the box never sees your
  instructions — you get an answer with nothing in it rather than an error.
- **`features` vs `opencv`.** Both run SIFT. `features` gives you one MATLAB-shaped
  `(2+128) x N` matrix per image and does no matching; `opencv` gives you separate
  `keypoints` / `descriptors` plus the matching and the fundamental matrix.
- **One image is allowed** on `opencv` — you get extraction only, and the matching fields
  are *absent*, not empty. Check before reading them.
- **`{"command": "reset"}`** is accepted by every box. Both of these are stateless, so it
  is a no-op here.


In [ ]:
print(features.run(config={"features": {"command": "reset"}}).config)
print(opencv.run(config={"opencv": {"command": "reset"}}).config)